# 02 - Preprocesamiento y preparación para modelado

Este cuaderno limpia los datos, construye características útiles y prepara los conjuntos de entrenamiento y prueba para los modelos predictivos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

DATA_PATH = Path("../data/who_ambient_air_quality_database_version_2024_(v6.1).xlsx")

# Cargar datos
raw_df = pd.read_excel(DATA_PATH, sheet_name="Update 2024 (V6.1)")

# Limpiar y transformar variables numéricas
for col in ["year", "population", "latitude", "longitude", "pm10_concentration", "pm25_concentration", "no2_concentration", "pm10_tempcov", "pm25_tempcov", "no2_tempcov", "who_ms"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Eliminar filas sin target
clean_df = raw_df.dropna(subset=["pm25_concentration"]).copy()

# Ajustes simples de calidad
clean_df["type_of_stations"] = clean_df["type_of_stations"].fillna("Unknown")
clean_df["country_name"] = clean_df["country_name"].fillna("Unknown")
clean_df["who_region"] = clean_df["who_region"].fillna("Unknown")

# Definir variables de entrada y objetivo
features = [
    "year", "population", "latitude", "longitude",
    "pm10_concentration", "no2_concentration",
    "pm10_tempcov", "no2_tempcov",
    "who_region", "country_name", "type_of_stations"
]
target = "pm25_concentration"

X = clean_df[features]
y = clean_df[target]

# Separación train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape}")
print(f"Prueba: {X_test.shape}")

X_train.head()

In [ ]:
# Definir preprocesador
numeric_features = ["year", "population", "latitude", "longitude", "pm10_concentration", "no2_concentration", "pm10_tempcov", "no2_tempcov"]
categorical_features = ["who_region", "country_name", "type_of_stations"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Forma de los datos procesados:")
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
# Mostrar nombres de columnas del preprocesador
feature_names = []
feature_names.extend(numeric_features)
feature_names.extend(preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features))

print(feature_names[:20])
print(f"Total de características: {len(feature_names)}")

In [ ]:
# Guardar una versión procesada para reutilizar en el siguiente notebook
processed_df = pd.DataFrame(X_train_processed.toarray(), columns=feature_names)
processed_df["target"] = y_train.reset_index(drop=True)
processed_df.head()